In [ ]:
# ===================================================================================
# WEB SCENE MIGRATION (3D Maps)
# - Clones Web Scene JSON from source portal
# - Rewires all operational layer references using migration ledger (ID_MAP)
# - Fixes layer URLs to point to target scene/feature services
# - Copies metadata, thumbnails, sharing, and owner/folder assignments
# - Logs all migrations to shared ledger (migration_history.csv)
#
# Follows same pattern as 2_Migrate_WebMaps.ipynb but for 3D Scene Viewer content.
# ===================================================================================

import pandas as pd
import json
import copy
import time
import csv
import os
import datetime
import re
import urllib3
import requests
import sys
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_12_web_scenes.json")
if os.path.exists(_sidecar_path):
    import json as _json
    WEB_SCENE_IDS_TO_MIGRATE = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(WEB_SCENE_IDS_TO_MIGRATE)} Web Scene IDs from sidecar.")
else:
    WEB_SCENE_IDS_TO_MIGRATE = [
        # Example Source ID's
        # "abc123def456...",
    ]

# =============================================================================
# --- CONNECT ------------------------------------------------------------------
# =============================================================================
source_gis = None
target_gis = None

print("Connecting...")
try:
    session = requests.Session()
    retry_strategy = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount('https://', adapter)

    source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL, session=session)
    target_gis = GIS(url=TARGET_URL, token=TARGET_TOKEN, verify_cert=False, referer=TARGET_URL, session=session)

    print(f"   > Source Connected: {source_gis.url}")
    if not target_gis.users.me: raise Exception("Target login failed.")
    print(f"   > Target Connected: {target_gis.users.me.username}")

except Exception as e:
    print(f"\n❌ CRITICAL CONNECTION FAILURE: {e}")
    sys.exit(1)

if not source_gis or not target_gis:
    print("❌ GIS Initialization incomplete. Aborting.")
    sys.exit(1)

# =============================================================================
# --- LOAD LEDGER --------------------------------------------------------------
# =============================================================================
ID_MAP = {}
if os.path.exists(LOG_FILE):
    try:
        df_log = pd.read_csv(LOG_FILE)
        if "SourceID" in df_log.columns and "TargetID" in df_log.columns:
            for _, row in df_log.iterrows():
                s_id = str(row["SourceID"]).strip()
                t_id = str(row["TargetID"]).strip()
                if s_id and t_id and s_id != "nan" and t_id != "nan":
                    ID_MAP[s_id] = t_id
        print(f"✅ Loaded Ledger: {len(ID_MAP)} ID mappings found.")
    except Exception as e:
        print(f"⚠️ Failed to load CSV: {e}")

STATS = {
    "scenes_scanned": 0,
    "scenes_created": 0,
    "failures": 0,
    "scenes_skipped_log": 0,
    "scenes_skipped_missing_data": 0,
    "layers_rewired": 0,
}

ALREADY_MIGRATED_IDS = set()
if os.path.exists(LOG_FILE):
    try:
        df_log = pd.read_csv(LOG_FILE)
        if "SourceID" in df_log.columns:
            ALREADY_MIGRATED_IDS = set(df_log["SourceID"].astype(str).str.strip())
    except: pass

def log_migration(source_id, index, target_id, title, item_type):
    try:
        with open(LOG_FILE, mode="a", newline="") as f:
            csv.writer(f).writerow([
                source_id, index, target_id, title,
                datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), item_type
            ])
            ALREADY_MIGRATED_IDS.add(str(source_id))
    except: pass

# =============================================================================
# --- ENSURE ADMIN FOLDER EXISTS -----------------------------------------------
# =============================================================================
ACTIVE_FOLDER = DEFAULT_FOLDER

def verify_admin_default_folder():
    global ACTIVE_FOLDER
    try:
        me = target_gis.users.me
        folder_names = []
        for f in me.folders:
            if isinstance(f, dict):
                folder_names.append(f.get('title'))
            elif hasattr(f, 'title'):
                folder_names.append(f.title)
        if DEFAULT_FOLDER not in folder_names:
            print(f"   [Setup] Creating missing admin folder: '{DEFAULT_FOLDER}'...")
            target_gis.content.create_folder(DEFAULT_FOLDER)
        else:
            print(f"   [Setup] Admin folder '{DEFAULT_FOLDER}' verified.")
    except Exception as e:
        print(f"   ⚠️ Warning: Could not verify/create admin folder. Falling back to ROOT. Error: {e}")
        ACTIVE_FOLDER = None

verify_admin_default_folder()

# =============================================================================
# --- HELPERS (METADATA, OWNER, FOLDERS, SHARING) ------------------------------
# =============================================================================
def copy_full_metadata(source_item, target_item, source_id):
    try:
        final_tags = list(source_item.tags or [])
        if "Migrated" not in final_tags: final_tags.append("Migrated")
        tag_marker = f"SourceID:{source_id}"
        if tag_marker not in final_tags: final_tags.append(tag_marker)
        update_props = {
            "title": target_item.title,
            "tags": final_tags,
            "snippet": source_item.snippet,
            "description": source_item.description,
            "accessInformation": source_item.accessInformation,
            "licenseInfo": source_item.licenseInfo
        }
        if getattr(source_item, "categories", None):
            update_props["categories"] = source_item.categories
        target_item.update(item_properties=update_props)
    except: pass

def get_source_folder_name(source_item):
    try:
        if not source_item.ownerFolder: return None
        user = source_gis.users.get(source_item.owner)
        if user:
            for f in user.folders:
                fid = f['id'] if isinstance(f, dict) else f.id
                ftitle = f['title'] if isinstance(f, dict) else f.title
                if fid == source_item.ownerFolder: return ftitle
    except: pass
    return None

def ensure_target_folder_exists(username, folder_name):
    try:
        user = target_gis.users.get(username)
        if not user: return False
        existing_folders = []
        for f in user.folders:
            if hasattr(f, 'title'): existing_folders.append(f.title)
            elif isinstance(f, dict): existing_folders.append(f.get('title'))
        if folder_name in existing_folders: return True
        print(f"      + Creating folder '{folder_name}' for user '{username}'...")
        target_gis.content.create_folder(folder_name, owner=username)
        return True
    except Exception as e:
        print(f"      ⚠ Folder creation error: {e}")
        return False

def assign_correct_owner_and_folder(source_item, target_item):
    try:
        source_owner = source_item.owner
        target_owner_to_use = DEFAULT_OWNER
        target_folder_to_use = ACTIVE_FOLDER
        found_user = target_gis.users.get(source_owner)
        if found_user:
            print(f"      👤 Source owner '{source_owner}' found in target.")
            target_owner_to_use = source_owner
            src_folder_name = get_source_folder_name(source_item)
            if src_folder_name:
                if ensure_target_folder_exists(target_owner_to_use, src_folder_name):
                    target_folder_to_use = src_folder_name
                else:
                    print(f"      ⚠️ Could not create folder '{src_folder_name}'. Using Default.")
            else:
                target_folder_to_use = None
        else:
            print(f"      👤 Source owner '{source_owner}' NOT found. Defaulting to '{DEFAULT_OWNER}'.")
        print(f"      📂 Moving to: Owner={target_owner_to_use}, Folder={target_folder_to_use}")
        target_item.reassign_to(target_owner_to_use, target_folder=target_folder_to_use)
    except Exception as e:
        print(f"      ⚠️ Reassign/Move Failed: {e} (Item remains with Creator)")

def copy_thumbnail(source_item, target_item):
    try:
        print("      🖼️ Copying Thumbnail...")
        temp_dir = "thumbnails_temp"
        os.makedirs(temp_dir, exist_ok=True)
        thumb_path = source_item.download_thumbnail(save_folder=temp_dir)
        if thumb_path: target_item.update(thumbnail=thumb_path)
    except: pass

def mirror_source_sharing(source_item, target_item):
    try:
        print("      👥 Mirroring Sharing Permissions (Source -> Target)...")
        is_public = False
        is_org = False
        if source_item.access == 'public':
            is_public = True
            is_org = True
        elif source_item.access == 'org':
            is_org = True

        source_groups = []
        try:
            raw_groups = source_item.sharing.groups
            if isinstance(raw_groups, list):
                source_groups = raw_groups
            else:
                raise ValueError("Not a list")
        except:
            raw_shared = source_item.shared_with
            if isinstance(raw_shared, dict) and 'groups' in raw_shared:
                source_groups = raw_shared['groups']
            elif isinstance(raw_shared, list):
                source_groups = raw_shared

        target_group_ids = []
        if source_groups:
            print(f"         - Found {len(source_groups)} source groups. Mapping...")
            for sg in source_groups:
                sg_title = sg.title if hasattr(sg, 'title') else str(sg)
                found_groups = target_gis.groups.search(f"title:\"{sg_title}\"")
                matched_group = next((g for g in found_groups if g.title == sg_title), None)
                if matched_group:
                    target_group_ids.append(matched_group.id)
                    print(f"           + Mapped: '{sg_title}'")
                else:
                    print(f"           - Skipped: '{sg_title}' (Not found in Target)")

        target_item.share(everyone=is_public, org=is_org, groups=target_group_ids)
        print(f"         ✅ Shared (Org={is_org}, Public={is_public}, Groups={len(target_group_ids)})")
    except Exception as e:
        print(f"      ⚠ Sharing Mirror Failed: {e}")

# =============================================================================
# --- CORE LOGIC: REWIRE WEB SCENE JSON ----------------------------------------
# =============================================================================
CACHE_TARGET_URLS = {}

def get_target_service_url(target_id):
    if target_id in CACHE_TARGET_URLS: return CACHE_TARGET_URLS[target_id]
    try:
        item = target_gis.content.get(target_id)
        if item:
            url = item.url
            CACHE_TARGET_URLS[target_id] = url
            return url
    except: pass
    return None

def rewire_scene_layer(layer):
    """Rewire a single layer dict in a Web Scene JSON, checking ID_MAP for matching services."""
    item_id = layer.get("itemId", "")
    url = layer.get("url", "")
    title = layer.get("title", "(no title)")

    # Try to match by itemId first
    if item_id and item_id in ID_MAP:
        target_id = ID_MAP[item_id]
        target_url = get_target_service_url(target_id)
        if target_url:
            layer["itemId"] = target_id
            # Preserve sublayer index if present
            if "/SceneServer/" in url:
                try:
                    parts = url.split("/SceneServer/")
                    suffix = parts[1] if len(parts) > 1 else ""
                    layer["url"] = f"{target_url}/{suffix}" if suffix else target_url
                except:
                    layer["url"] = target_url
            elif "/FeatureServer/" in url:
                try:
                    idx = url.split("/FeatureServer/")[-1]
                    layer["url"] = f"{target_url}/{idx}" if idx.isdigit() else target_url
                except:
                    layer["url"] = target_url
            elif "/MapServer/" in url:
                try:
                    idx = url.split("/MapServer/")[-1]
                    layer["url"] = f"{target_url}/{idx}" if idx.isdigit() else target_url
                except:
                    layer["url"] = target_url
            else:
                layer["url"] = target_url

            if "serviceToken" in layer: del layer["serviceToken"]
            STATS["layers_rewired"] += 1
            print(f"     ✅ Rewired: {title} -> {target_id}")
            return

    # Fallback: try to find source item by URL search
    if url and ENTERPRISE_HOST in url:
        for server_type in ["/SceneServer", "/FeatureServer", "/MapServer"]:
            if server_type in url:
                base_url = url.split(server_type)[0] + server_type
                try:
                    items = source_gis.content.search(f'url:"{base_url}"', max_items=1)
                    if items:
                        src_id = items[0].id
                        if src_id in ID_MAP:
                            target_id = ID_MAP[src_id]
                            target_url = get_target_service_url(target_id)
                            if target_url:
                                layer["itemId"] = target_id
                                layer["url"] = target_url
                                if "serviceToken" in layer: del layer["serviceToken"]
                                STATS["layers_rewired"] += 1
                                print(f"     ✅ Rewired (URL match): {title} -> {target_id}")
                                return
                except: pass
                break

    # Not an enterprise layer or no mapping found
    if url and ENTERPRISE_HOST in url.lower():
        print(f"     ⚠️ No mapping for enterprise layer: {title} (itemId={item_id})")
    else:
        pass  # External/AGOL layer, leave as-is

def process_scene_layers(layer_list):
    """Recursively process all layers in a Web Scene JSON."""
    if not isinstance(layer_list, list):
        return
    for layer in layer_list:
        if isinstance(layer, dict):
            rewire_scene_layer(layer)
            # Handle nested layer groups
            if "layers" in layer:
                process_scene_layers(layer["layers"])

def deep_id_swap(scene_json):
    """
    Deep scan the entire Web Scene JSON for 32-char hex IDs and swap any
    that appear in the ID_MAP. This catches IDs in slides, popups, widgets, etc.
    """
    json_str = json.dumps(scene_json)
    hex_ids = set(re.findall(r'\b[0-9a-f]{32}\b', json_str))

    swapped = 0
    for old_id in hex_ids:
        if old_id in ID_MAP:
            new_id = ID_MAP[old_id]
            json_str = json_str.replace(old_id, new_id)
            swapped += 1

    if swapped > 0:
        print(f"     🔄 Deep ID swap: replaced {swapped} unique IDs in scene JSON")

    return json.loads(json_str)

# =============================================================================
# --- MAIN LOOP ----------------------------------------------------------------
# =============================================================================
print(f"\nStarting Web Scene Migration...")
start_time = time.time()

for scene_id in WEB_SCENE_IDS_TO_MIGRATE:
    try:
        STATS["scenes_scanned"] += 1

        if str(scene_id) in ALREADY_MIGRATED_IDS:
            print(f"\n[Skip] Scene {scene_id} found in History Log.")
            STATS["scenes_skipped_log"] += 1
            continue

        source_scene = source_gis.content.get(scene_id)
        if not source_scene:
            print(f"❌ Scene {scene_id} not found.")
            STATS["scenes_skipped_missing_data"] += 1
            STATS["failures"] += 1
            continue

        print(f"\nProcessing Scene: {source_scene.title}...")

        target_title = str(source_scene.title) if source_scene.title else "Untitled Scene"
        if APPEND_MIGRATED:
            target_title = f"{target_title} (Migrated)"

        # Check if already exists in target
        existing = target_gis.content.search(f'title:"{target_title}" type:"Web Scene"', max_items=1)
        if existing:
            print(f"   ⚠️ Scene already exists in Target. Skipping.")
            STATS["scenes_skipped_log"] += 1
            continue

        # Get scene JSON data
        scene_json = source_scene.get_data()
        if not scene_json:
            print(f"   ❌ Could not retrieve scene JSON. Skipping.")
            STATS["scenes_skipped_missing_data"] += 1
            STATS["failures"] += 1
            continue

        new_json = copy.deepcopy(scene_json)

        # Rewire operational layers
        if "operationalLayers" in new_json:
            print("   1. Rewiring operational layers...")
            process_scene_layers(new_json["operationalLayers"])

        # Rewire base map layers if present
        if "baseMap" in new_json and isinstance(new_json.get("baseMap"), dict):
            if "baseMapLayers" in new_json["baseMap"]:
                process_scene_layers(new_json["baseMap"]["baseMapLayers"])

        # Rewire ground layers if present (elevation layers)
        if "ground" in new_json and isinstance(new_json.get("ground"), dict):
            if "layers" in new_json["ground"]:
                process_scene_layers(new_json["ground"]["layers"])

        # Deep ID swap for anything embedded (slides, popups, widgets)
        new_json = deep_id_swap(new_json)

        # Create the new Web Scene item
        print("   2. Creating Web Scene item...")
        snippet_val = str(source_scene.snippet) if source_scene.snippet else ""
        tags_list = ["Migrated", f"SourceID:{scene_id}"]
        tags_str = ",".join(tags_list)

        item_props = {
            "type": "Web Scene",
            "title": target_title,
            "snippet": snippet_val,
            "tags": tags_str,
            "text": json.dumps(new_json)
        }

        new_scene = target_gis.content.add(item_props, folder=ACTIVE_FOLDER)

        if new_scene:
            copy_full_metadata(source_scene, new_scene, scene_id)
            copy_thumbnail(source_scene, new_scene)
            mirror_source_sharing(source_scene, new_scene)
            assign_correct_owner_and_folder(source_scene, new_scene)

            log_migration(scene_id, "N/A", new_scene.id, new_scene.title, "Web Scene")
            STATS["scenes_created"] += 1
            print(f"🚀 SUCCESS: Created Scene '{new_scene.title}'")

        time.sleep(THROTTLE_SECONDS)

    except Exception as e:
        print(f"❌ Failed processing scene {scene_id}: {e}")
        STATS["failures"] += 1

# =============================================================================
# --- FINAL REPORT -------------------------------------------------------------
# =============================================================================
end_time = time.time()
total_seconds = int(end_time - start_time)
duration_str = f"{total_seconds // 3600}h {(total_seconds % 3600) // 60}m {total_seconds % 60}s"

print("\n" + "=" * 60)
print("        WEB SCENE MIGRATION REPORT")
print("=" * 60)
print(f" ⏱️  Total Duration:             {duration_str}")
print("-" * 60)
print(f" 🌐 Scenes Scanned:             {STATS['scenes_scanned']}")
print(f" 🧾 Skipped (log):              {STATS['scenes_skipped_log']}")
print(f" 🛑 Skipped (missing data):     {STATS['scenes_skipped_missing_data']}")
print(f" ✅ Scenes Created:             {STATS['scenes_created']}")
print(f" 🔗 Layers Rewired:             {STATS['layers_rewired']}")
print(f" ❌ Failures:                   {STATS['failures']}")
print("=" * 60)